In [ ]:
!pip install -q fastapi uvicorn sqlalchemy aiosqlite pydantic[email] nest-asyncio pyngrok

In [ ]:
import asyncio
import nest_asyncio

nest_asyncio.apply()

from datetime import datetime

from fastapi import FastAPI, Depends, HTTPException
from fastapi.responses import JSONResponse

from pydantic import BaseModel, Field, EmailStr, ConfigDict, field_validator, model_validator

from sqlalchemy import String, Float, ForeignKey, DateTime, select
from sqlalchemy.ext.asyncio import (
    create_async_engine,
    AsyncSession,
    async_sessionmaker
)
from sqlalchemy.orm import (
    DeclarativeBase,
    Mapped,
    mapped_column,
    relationship
)

In [ ]:
DATABASE_URL = "sqlite+aiosqlite:///./day5.db"

engine = create_async_engine(
    DATABASE_URL,
    echo=True
)

AsyncSessionLocal = async_sessionmaker(
    engine,
    expire_on_commit=False
)


class Base(DeclarativeBase):
    pass


async def get_db():
    async with AsyncSessionLocal() as session:
        yield session

In [ ]:
class User(Base):
    __tablename__ = "users"

    id: Mapped[int] = mapped_column(
        primary_key=True,
        index=True
    )

    name: Mapped[str] = mapped_column(
        String(100)
    )

    email: Mapped[str] = mapped_column(
        String(150),
        unique=True,
        index=True
    )

    created_at: Mapped[datetime] = mapped_column(
        DateTime,
        default=datetime.utcnow
    )

    products: Mapped[list["Product"]] = relationship(
        back_populates="owner",
        cascade="all, delete-orphan"
    )


class Product(Base):
    __tablename__ = "products"

    id: Mapped[int] = mapped_column(
        primary_key=True,
        index=True
    )

    name: Mapped[str] = mapped_column(
        String(150)
    )

    price: Mapped[float] = mapped_column(
        Float
    )

    user_id: Mapped[int] = mapped_column(
        ForeignKey("users.id")
    )

    owner: Mapped["User"] = relationship(
        back_populates="products"
    )

In [ ]:
async def create_tables():
    async with engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)


await create_tables()

print("Database tables created successfully!")

2026-09-22 09:19:33,837 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-09-22 09:19:33,840 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("users")


INFO:sqlalchemy.engine.Engine:PRAGMA main.table_info("users")


2026-09-22 09:19:33,843 INFO sqlalchemy.engine.Engine [raw sql] ()


...

2026-09-22 09:19:33,852 INFO sqlalchemy.engine.Engine [raw sql] ()


INFO:sqlalchemy.engine.Engine:[raw sql] ()


2026-09-22 09:19:33,857 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Database tables created successfully!


In [ ]:
class UserCreate(BaseModel):

    name: str = Field(
        min_length=2,
        max_length=100
    )

    email: EmailStr

    @field_validator("name")
    @classmethod
    def validate_name(cls, value):

        if not value.strip():
            raise ValueError("Name cannot be empty")

        return value.strip()


class UserUpdate(BaseModel):

    name: str | None = Field(
        default=None,
        min_length=2,
        max_length=100
    )

    email: EmailStr | None = None


class UserResponse(BaseModel):

    model_config = ConfigDict(
        from_attributes=True
    )

    id: int
    name: str
    email: EmailStr

In [ ]:
class ProductCreate(BaseModel):

    name: str = Field(
        min_length=2,
        max_length=150
    )

    price: float = Field(
        gt=0
    )

    user_id: int = Field(
        gt=0
    )


class ProductUpdate(BaseModel):

    name: str | None = Field(
        default=None,
        min_length=2,
        max_length=150
    )

    price: float | None = Field(
        default=None,
        gt=0
    )


class ProductResponse(BaseModel):

    model_config = ConfigDict(
        from_attributes=True
    )

    id: int
    name: str
    price: float
    user_id: int

In [ ]:
class ProductSimple(BaseModel):

    id: int
    name: str
    price: float

    model_config = ConfigDict(
        from_attributes=True
    )


class UserWithProducts(BaseModel):

    id: int
    name: str
    email: EmailStr

    products: list[ProductSimple] = []

    model_config = ConfigDict(
        from_attributes=True
    )

In [ ]:
class PasswordRequest(BaseModel):

    password: str = Field(min_length=6)

    confirm_password: str = Field(min_length=6)

    @model_validator(mode="after")
    def check_passwords(self):

        if self.password != self.confirm_password:
            raise ValueError("Passwords do not match")

        return self

In [ ]:
try:
    PasswordRequest(
        password="abcdef",
        confirm_password="abcdef"
    )

    print("Password validation successful!")

except Exception as e:
    print(e)

Password validation successful!


In [ ]:
app = FastAPI(
    title="Day 5 FastAPI CRUD",
    description="FastAPI + Pydantic V2 + SQLAlchemy 2.0",
    version="1.0.0"
)


@app.get("/")
async def root():

    return {
        "message": "Day 5 FastAPI application is running!"
    }

In [ ]:
@app.post(
    "/users/",
    response_model=UserResponse,
    tags=["Users"]
)
async def create_user(
    user_data: UserCreate,
    db: AsyncSession = Depends(get_db)
):

    # Check if email already exists
    result = await db.execute(
        select(User).where(
            User.email == user_data.email
        )
    )

    existing_user = result.scalar_one_or_none()

    if existing_user:
        raise HTTPException(
            status_code=400,
            detail="Email already registered"
        )

    user = User(
        name=user_data.name,
        email=user_data.email
    )

    db.add(user)

    await db.commit()

    await db.refresh(user)

    return user

In [ ]:
@app.get(
    "/users/",
    response_model=list[UserResponse],
    tags=["Users"]
)
async def get_users(
    db: AsyncSession = Depends(get_db)
):

    result = await db.execute(
        select(User)
    )

    users = result.scalars().all()

    return users

In [ ]:
@app.get(
    "/users/{user_id}",
    response_model=UserResponse,
    tags=["Users"]
)
async def get_user(
    user_id: int,
    db: AsyncSession = Depends(get_db)
):

    result = await db.execute(
        select(User).where(
            User.id == user_id
        )
    )

    user = result.scalar_one_or_none()

    if user is None:

        raise HTTPException(
            status_code=404,
            detail="User not found"
        )

    return user

In [ ]:
@app.put(
    "/users/{user_id}",
    response_model=UserResponse,
    tags=["Users"]
)
async def update_user(
    user_id: int,
    user_data: UserUpdate,
    db: AsyncSession = Depends(get_db)
):

    result = await db.execute(
        select(User).where(
            User.id == user_id
        )
    )

    user = result.scalar_one_or_none()

    if user is None:

        raise HTTPException(
            status_code=404,
            detail="User not found"
        )

    if user_data.name is not None:
        user.name = user_data.name

    if user_data.email is not None:
        user.email = user_data.email

    await db.commit()

    await db.refresh(user)

    return user

In [ ]:
@app.delete(
    "/users/{user_id}",
    tags=["Users"]
)
async def delete_user(
    user_id: int,
    db: AsyncSession = Depends(get_db)
):

    result = await db.execute(
        select(User).where(
            User.id == user_id
        )
    )

    user = result.scalar_one_or_none()

    if user is None:

        raise HTTPException(
            status_code=404,
            detail="User not found"
        )

    await db.delete(user)

    await db.commit()

    return {
        "message": "User deleted successfully"
    }

In [ ]:
@app.post(
    "/products/",
    response_model=ProductResponse,
    tags=["Products"]
)
async def create_product(
    product_data: ProductCreate,
    db: AsyncSession = Depends(get_db)
):

    # Check whether user exists
    result = await db.execute(
        select(User).where(
            User.id == product_data.user_id
        )
    )

    user = result.scalar_one_or_none()

    if user is None:

        raise HTTPException(
            status_code=404,
            detail="User not found"
        )

    product = Product(
        name=product_data.name,
        price=product_data.price,
        user_id=product_data.user_id
    )

    db.add(product)

    await db.commit()

    await db.refresh(product)

    return product

In [ ]:
@app.get(
    "/products/",
    response_model=list[ProductResponse],
    tags=["Products"]
)
async def get_products(
    db: AsyncSession = Depends(get_db)
):

    result = await db.execute(
        select(Product)
    )

    products = result.scalars().all()

    return products

In [ ]:
@app.get(
    "/products/{product_id}",
    response_model=ProductResponse,
    tags=["Products"]
)
async def get_product(
    product_id: int,
    db: AsyncSession = Depends(get_db)
):

    result = await db.execute(
        select(Product).where(
            Product.id == product_id
        )
    )

    product = result.scalar_one_or_none()

    if product is None:

        raise HTTPException(
            status_code=404,
            detail="Product not found"
        )

    return product

In [ ]:
@app.put(
    "/products/{product_id}",
    response_model=ProductResponse,
    tags=["Products"]
)
async def update_product(
    product_id: int,
    product_data: ProductUpdate,
    db: AsyncSession = Depends(get_db)
):

    result = await db.execute(
        select(Product).where(
            Product.id == product_id
        )
    )

    product = result.scalar_one_or_none()

    if product is None:

        raise HTTPException(
            status_code=404,
            detail="Product not found"
        )

    if product_data.name is not None:
        product.name = product_data.name

    if product_data.price is not None:
        product.price = product_data.price

    await db.commit()

    await db.refresh(product)

    return product

In [ ]:
@app.delete(
    "/products/{product_id}",
    tags=["Products"]
)
async def delete_product(
    product_id: int,
    db: AsyncSession = Depends(get_db)
):

    result = await db.execute(
        select(Product).where(
            Product.id == product_id
        )
    )

    product = result.scalar_one_or_none()

    if product is None:

        raise HTTPException(
            status_code=404,
            detail="Product not found"
        )

    await db.delete(product)

    await db.commit()

    return {
        "message": "Product deleted successfully"
    }

In [ ]:
for route in app.routes:
    print(
        route.methods,
        route.path
    )

{'HEAD', 'GET'} /openapi.json
{'HEAD', 'GET'} /docs
{'HEAD', 'GET'} /docs/oauth2-redirect
{'HEAD', 'GET'} /redoc
{'GET'} /
{'POST'} /users/
{'GET'} /users/
{'GET'} /users/{user_id}
{'PUT'} /users/{user_id}
{'DELETE'} /users/{user_id}
{'POST'} /products/
{'GET'} /products/
{'GET'} /products/{product_id}
{'PUT'} /products/{product_id}
{'DELETE'} /products/{product_id}


In [ ]:
# FASTAPI SERVER ERROR FIX — Colab

import asyncio
import threading
import time
import requests
import uvicorn
from google.colab.output import eval_js
from IPython.display import display, HTML

PORT = 8000

def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    config = uvicorn.Config(
        app,
        host="0.0.0.0",
        port=PORT,
        log_level="info"
    )

    server = uvicorn.Server(config)

    try:
        loop.run_until_complete(server.serve())
    except Exception as e:
        print("Server error:", e)
    finally:
        loop.close()

# Start FastAPI in background
server_thread = threading.Thread(
    target=run_server,
    daemon=True
)
server_thread.start()

# Give the server time to start
time.sleep(3)

# Check whether FastAPI is running
try:
    response = requests.get(
        f"http://127.0.0.1:{PORT}/",
        timeout=5
    )

    print("✅ FastAPI server started!")
    print("Status:", response.status_code)
    print("Response:", response.json())

    # Colab public/proxy URL
    url = eval_js(
        f"google.colab.kernel.proxyPort({PORT})"
    )

    print("\n🚀 Swagger UI:")
    print(url + "/docs")

    display(
        HTML(
            f'<a href="{url}/docs" target="_blank">'
            '<button style="font-size:18px;padding:10px;">'
            'Open FastAPI Swagger UI'
            '</button></a>'
        )
    )

except Exception as e:
    print("❌ FastAPI server could not start.")
    print("Error:", e)

INFO:     Started server process [7740]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:58006 - "GET / HTTP/1.1" 200 OK
✅ FastAPI server started!
Status: 200
Response: {'message': 'Day 5 FastAPI application is running!'}

🚀 Swagger UI:
https://8000-m-s-kkb-usc1c2-2twoc8tcbzxys-c.us-central1-2.prod.colab.dev/docs


In [ ]:
# ============================================================
# FASTAPI SERVER — CLEAN RESTART
# ============================================================

import asyncio
import threading
import time
import requests
import uvicorn

from google.colab.output import eval_js
from IPython.display import display, HTML

PORT = 8000


# ------------------------------------------------------------
# Start Uvicorn using the EXISTING FastAPI app
# ------------------------------------------------------------

def start_fastapi():
    try:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)

        config = uvicorn.Config(
            app=app,
            host="0.0.0.0",
            port=PORT,
            log_level="info"
        )

        server = uvicorn.Server(config)

        loop.run_until_complete(server.serve())

    except Exception as e:
        print("❌ Uvicorn error:")
        print(e)


# ------------------------------------------------------------
# Start server in background thread
# ------------------------------------------------------------

server_thread = threading.Thread(
    target=start_fastapi,
    daemon=True
)

server_thread.start()

# Wait for server
time.sleep(5)


# ------------------------------------------------------------
# Get Colab proxy URL
# ------------------------------------------------------------

base_url = eval_js(
    f"google.colab.kernel.proxyPort({PORT})"
)

print("========================================")
print("FASTAPI SERVER")
print("========================================")
print("API URL :", base_url)
print("Swagger :", base_url + "/docs")


# ------------------------------------------------------------
# Test OpenAPI
# ------------------------------------------------------------

try:

    response = requests.get(
        "http://127.0.0.1:8000/openapi.json",
        timeout=10
    )

    print("\nLocal OpenAPI status:", response.status_code)

    if response.status_code == 200:

        print("✅ FastAPI is running correctly!")

        openapi_data = response.json()
        paths = openapi_data.get("paths", {})

        print("\n========================================")
        print("AVAILABLE ROUTES")
        print("========================================")

        for path, methods in paths.items():

            for method in methods:

                print(
                    f"{method.upper():<10} {path}"
                )

        print("\n========================================")
        print("SWAGGER UI")
        print("========================================")

        display(
            HTML(
                f'''
                <a href="{base_url}/docs"
                   target="_blank">
                    <button style="
                        font-size:18px;
                        padding:12px 20px;
                        cursor:pointer;">
                        🚀 Open FastAPI Swagger
                    </button>
                </a>
                '''
            )
        )

    else:

        print("❌ FastAPI returned:", response.status_code)
        print("Response:", response.text)

except Exception as e:

    print("\n❌ FastAPI server could not start.")
    print("Error:", e)

INFO:     Started server process [7740]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): [errno 98] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


FASTAPI SERVER
API URL : https://8000-m-s-kkb-usc1c2-2twoc8tcbzxys-c.us-central1-2.prod.colab.dev
Swagger : https://8000-m-s-kkb-usc1c2-2twoc8tcbzxys-c.us-central1-2.prod.colab.dev/docs
INFO:     127.0.0.1:59864 - "GET /openapi.json HTTP/1.1" 200 OK

Local OpenAPI status: 200
✅ FastAPI is running correctly!

AVAILABLE ROUTES
GET        /
GET        /users/
POST       /users/
GET        /users/{user_id}
PUT        /users/{user_id}
DELETE     /users/{user_id}
GET        /products/
POST       /products/
GET        /products/{product_id}
PUT        /products/{product_id}
DELETE     /products/{product_id}

SWAGGER UI
